# Milestone 3: BPR

Starter notebook for milestone 3 of [Project - MovieLens Recommender](Project%20-%20MovieLens%20Recommender.md).

Cells marked **TODO** are yours to fill in; everything else runs as-is. Notebook is saved without outputs.

In [ ]:
import numpy as np, pandas as pd, pathlib, urllib.request, zipfile, io

DATA = pathlib.Path("ml-latest-small")
if not DATA.exists():
    # Or download manually from https://grouplens.org/datasets/movielens/ (ml-latest-small) and unzip here
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")
ratings = pd.read_csv(DATA / "ratings.csv")   # userId, movieId, rating, timestamp
movies = pd.read_csv(DATA / "movies.csv")     # movieId, title, genres
links = pd.read_csv(DATA / "links.csv")       # movieId, imdbId, tmdbId
ratings.head()

In [ ]:
# Leave-last-out split by time: for every user, the most recent positive is the test item,
# the one before it is validation. Positives = rating >= 4 (implicit feedback).
pos = ratings[ratings.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)   # 0 = most recent
test  = pos[pos["rank"] == 0][["userId", "movieId"]]
val   = pos[pos["rank"] == 1][["userId", "movieId"]]
train = pos[pos["rank"] >= 2][["userId", "movieId"]]

# Contiguous ids so we can index embedding tables later
uid = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}
iid = {m: i for i, m in enumerate(sorted(ratings.movieId.unique()))}
n_users, n_items = len(uid), len(iid)
print(n_users, n_items, len(train), len(val), len(test))

In [ ]:
def evaluate(score_fn, held_out, train, k=10, n_neg=99, seed=0):
    """HitRate@k and NDCG@k with 1 held-out positive vs n_neg sampled unseen items (sampled protocol).
    score_fn(user_id, item_ids) -> array of scores. TODO: switch to full ranking once everything works,
    sampled metrics can reorder models (see Evaluating Recommenders)."""
    rng = np.random.default_rng(seed)
    seen = train.groupby("userId").movieId.apply(set).to_dict()
    all_items = np.array(list(iid))
    hits, ndcg = [], []
    for u, m in held_out.itertuples(index=False):
        banned = seen.get(u, set()) | {m}
        negs = rng.choice(all_items, size=n_neg * 2, replace=False)
        negs = [x for x in negs if x not in banned][:n_neg]
        cand = np.array([m] + negs)
        s = np.asarray(score_fn(u, cand))
        rank = (s > s[0]).sum()          # 0 = positive ranked first (ties are optimistic: break them!)
        hits.append(rank < k)
        ndcg.append(1 / np.log2(rank + 2) if rank < k else 0.0)
    return {"HR@k": float(np.mean(hits)), "NDCG@k": float(np.mean(ndcg))}

## 1. Triple sampler
For each training positive $(u, i)$ sample a negative $j$ the user has not interacted with. BPR maximises $\ln\sigma(\hat x_{ui}-\hat x_{uj})$.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

u_tr = torch.tensor(train.userId.map(uid).to_numpy()); i_tr = torch.tensor(train.movieId.map(iid).to_numpy())
user_pos = train.groupby(train.userId.map(uid)).movieId.apply(lambda s: set(s.map(iid))).to_dict()

def sample_negatives(users, n_items):
    # TODO: uniformly sample j and resample while j in user_pos[u]. Vectorise if it is slow.
    raise NotImplementedError

In [ ]:
class BPRMF(nn.Module):
    def __init__(self, n_users, n_items, k=64):
        super().__init__()
        self.U = nn.Embedding(n_users, k); self.I = nn.Embedding(n_items, k); self.b = nn.Embedding(n_items, 1)
        nn.init.normal_(self.U.weight, std=0.01); nn.init.normal_(self.I.weight, std=0.01); nn.init.zeros_(self.b.weight)

    def score(self, u, i):
        return (self.U(u) * self.I(i)).sum(-1) + self.b(i).squeeze(-1)

def bpr_loss(model, u, i, j, reg=1e-4):
    raise NotImplementedError  # TODO: -log sigmoid(score(u,i) - score(u,j)).mean() + L2 on the embeddings used in the batch

## 2. Train and compare (TODO)
Adam, batch size 1024, ~30 epochs; evaluate on `val` every few epochs. Compare against M2 (same split, same `evaluate`).

Experiments: popularity-weighted negative sampling; more negatives per positive; the item bias term on/off.

In [ ]:
# TODO: training loop + evaluation curve